## Setup
Import libraries and set paths.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

RESULTS = Path("../results")
sns.set_theme(style="whitegrid", palette="muted")

## DESeq2 Results
Load the differential expression table output by DESeq2.

In [ ]:
deg = pd.read_csv(RESULTS / "tables/deseq2_results.tsv", sep="\t")
print(f"Shape: {deg.shape}")
print(deg["direction"].value_counts())
deg.head()

## Volcano Plot (Python/matplotlib)
Recreate the volcano plot to demonstrate Python-R interoperability.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors = {"Upregulated": "#D85A30", "Downregulated": "#378ADD", "Not significant": "#AAAAAA"}
for group, subset in deg.dropna(subset=["padj"]).groupby("direction"):
    ax.scatter(subset["log2FoldChange"], -np.log10(subset["padj"]),
               c=colors[group], s=6, alpha=0.6, label=group)
ax.axvline(x=1,  linestyle="--", color="grey", linewidth=0.8)
ax.axvline(x=-1, linestyle="--", color="grey", linewidth=0.8)
ax.axhline(y=-np.log10(0.05), linestyle="--", color="grey", linewidth=0.8)
ax.set_xlabel("log₂ Fold Change")
ax.set_ylabel("−log₁₀ Adjusted p-value")
ax.set_title("Tumor vs. Normal — Differential Expression")
ax.legend(markerscale=3)
plt.tight_layout()
plt.savefig(RESULTS / "figures/volcano_python.png", dpi=150)
plt.show()

## Sample Correlation Heatmap
Pearson correlation of VST-normalized counts across all samples.

In [ ]:
norm = pd.read_csv(RESULTS / "tables/normalized_counts.tsv", sep="\t", index_col=0)
corr = norm.corr(method="pearson")
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r",
            center=1.0, vmin=0.85, vmax=1.0,
            linewidths=0.5, ax=ax)
ax.set_title("Sample-to-Sample Pearson Correlation (VST counts)")
plt.tight_layout()
plt.savefig(RESULTS / "figures/sample_correlation.png", dpi=150)
plt.show()

## Genes of Interest
Expression of known breast cancer marker genes across conditions.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, "..")

genes_of_interest = ["MKI67", "ESR1", "ERBB2", "TP53"]
samples_meta = pd.read_csv("../config/samples.tsv", sep="\t", index_col=0)

# Filter normalized counts to genes of interest
subset = norm.loc[norm.index.isin(genes_of_interest)].T
subset = subset.join(samples_meta[["condition"]])

fig, axes = plt.subplots(1, len(genes_of_interest), figsize=(14, 5), sharey=False)
for ax, gene in zip(axes, genes_of_interest):
    if gene not in subset.columns:
        ax.set_title(f"{gene}\n(not detected)")
        ax.axis("off")
        continue
    sns.boxplot(data=subset, x="condition", y=gene, ax=ax,
                palette={"tumor": "#D85A30", "normal": "#378ADD"},
                width=0.5)
    sns.stripplot(data=subset, x="condition", y=gene, ax=ax,
                  color="black", size=4, jitter=True, alpha=0.6)
    ax.set_title(gene, fontsize=12, fontweight="bold")
    ax.set_xlabel("")
fig.suptitle("Marker Gene Expression: Tumor vs. Normal", fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS / "figures/marker_genes.png", dpi=150)
plt.show()

## DEG Summary
Key numbers for the README results table.

In [ ]:
sig = deg[deg["significant"] == True]
print(f"Total DEGs:       {len(sig)}")
print(f"Upregulated:      {(sig['direction']=='Upregulated').sum()}")
print(f"Downregulated:    {(sig['direction']=='Downregulated').sum()}")
print(f"\nTop 10 by adjusted p-value:")
print(sig.nsmallest(10, "padj")[["gene_id","log2FoldChange","padj","direction"]].to_string(index=False))